In [1]:
%pip install transformers tokenizers datasets evaluate accelerate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
print(f"Working dir: {WORKING_DIR}")

In [2]:
from datasets import load_dataset

dataset = load_dataset("wikimedia/wikipedia", "20231101.he")
print(dataset)
print(dataset["train"][0]["text"][:500])

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 333874
    })
})
מָתֵמָטִיקָה היא תחום דעת העוסק במושגים כגון כמות, מבנה, מרחב ושינוי. המתמטיקאים מחפשים דפוסים ותבניות משותפות במספרים, במרחב, במדע ובהפשטות דמיוניות.

המתמטיקה התפתחה ממנייה, חישוב ומדידה ומהמחקר השיטתי של צורות ותנועה של עצמים מוחשיים. הידע והשימוש במתמטיקה בסיסית היוו תמיד חלק טבעי וחיוני בחיי האדם והקבוצה. ניתן למצוא שכלולים של הרעיונות הבסיסיים בטקסטים המתמטיים שהגו המצרים, הבבלים, ההודים, הסינים, היוונים והמוסלמים. כבר בשלב מוקדם בלטו שלושה מאפיינים המלווים את המתמטיקה עד היום:
 הפשטה: אף 


In [ ]:
import tokenizers as tok

tokenizer = tok.Tokenizer(tok.models.BPE())
tokenizer.pre_tokenizer = tok.pre_tokenizers.Whitespace()
tokenizer.decoder = tok.decoders.BPEDecoder()

trainer = tok.trainers.BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=["<|pad|>", "<|endoftext|>"]
)

tokenizer.train_from_iterator(
    (text for text in dataset["train"]["text"] if text.strip()),
    trainer=trainer
)

tokenizer.save(f"{WORKING_DIR}/hebrew_bpe_tokenizer.json")
print("Tokenizer trained and saved.")
print("Vocab size:", tokenizer.get_vocab_size())

sample = "שלום עולם, זהו מבחן לטוקנייזר העברי."
encoded = tokenizer.encode(sample)
print("Sample:", sample)
print("Tokens:", encoded.tokens)
print("IDs:   ", encoded.ids)

In [ ]:
import transformers as tr

vocab = tokenizer.get_vocab()
ttokenizer = tr.PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="<|pad|>",
    pad_token="<|pad|>",
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>"
)
len(vocab)

In [ ]:
config = tr.GPT2Config(
    vocab_size=len(vocab),
    n_positions=256,
    n_embd=256,
    n_layer=6,
    n_head=8,
    bos_token_id=ttokenizer.bos_token_id,
    eos_token_id=ttokenizer.eos_token_id,
    pad_token_id=ttokenizer.pad_token_id
)
print(config)

In [ ]:
gpt = tr.GPT2LMHeadModel(config)
print(gpt)
print(f"\nTotal parameters: {sum(p.numel() for p in gpt.parameters()):,}")

In [ ]:
res = gpt.generate(
    **ttokenizer("Мне нравится ", return_tensors="pt"),
    max_new_tokens=50,
    top_k=3,
    do_sample=True
)
ttokenizer.decode(res[0])

In [ ]:
import torch
from itertools import chain
from datasets import load_from_disk

BLOCK_SIZE = 256
DATASET_PATH = f"{WORKING_DIR}/hebrew_tokenized"

if os.path.exists(DATASET_PATH):
    lm_dataset = load_from_disk(DATASET_PATH)
    lm_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    print(f"Loaded from disk: {len(lm_dataset):,} chunks")
else:
    def tokenize(examples):
        return ttokenizer(examples["text"], truncation=False, padding=False)

    def group_texts(examples):
        concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
        total_length = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
        result = {
            k: [t[i:i+BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
            for k, t in concatenated.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    tokenized = dataset["train"].map(
        tokenize,
        batched=True,
        batch_size=1000,
        remove_columns=dataset["train"].column_names,
        desc="Tokenizing"
    )
    lm_dataset = tokenized.map(group_texts, batched=True, desc="Chunking")
    lm_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    lm_dataset.save_to_disk(DATASET_PATH)
    print(f"Tokenized and saved: {len(lm_dataset):,} chunks")

print(f"Sample input_ids shape: {lm_dataset[0]['input_ids'].shape}")

In [ ]:
targs = tr.TrainingArguments(
    output_dir=f"{WORKING_DIR}/gpt2-hebrew",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    save_steps=1500,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = tr.Trainer(
    model=gpt,
    args=targs,
    train_dataset=lm_dataset,
    data_collator=tr.default_data_collator
)

trainer.train()

trainer.save_model(f"{WORKING_DIR}/gpt2-hebrew/final")
ttokenizer.save_pretrained(f"{WORKING_DIR}/gpt2-hebrew/final")
print("Model saved.")